<a href="https://colab.research.google.com/github/spencerCrowell/WeatherRiskAnalysis/blob/main/model_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = 'lanwuxian'
os.environ['KAGGLE_KEY'] = 'KGAT_693be7f21d7dbf6c8c2bcefce9c6df58'

!pip install --upgrade --force-reinstall --no-deps kaggle

!kaggle datasets download -d "joeysoliman/madis-metar-visibility-2002"

# Add a check to verify the downloaded file's presence
print("\nContents of /content/ after Kaggle download:")
!ls -F /content/

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: kaggle
    Found existing installation: kaggle 2.0.0
    Uninstalling kaggle-2.0.0:
      Successfully uninstalled kaggle-2.0.0
Dataset URL: https://www.kaggle.com/datasets/joeysoliman/madis-metar-visibility-2002
License(s): CC0-1.0
100% 1.25G/1.25G [00:11<00:00, 122MB/s]


Contents of /content/ after Kaggle download:
madis-metar-visibility-2002.zip  sample_data/


### Scale Features and Target Variable

**Instructions:** Apply `MinMaxScaler` to scale the selected features and the target variable. It's crucial to fit the scaler only on the training data to prevent data leakage from the test set.

# Task
Unzip the file "madis-metar-visibility-2002.zip".

## Unzip Dataset

### Subtask:
Unzip the downloaded 'madis-metar-visibility-2002.zip' file to access the visibility data.


**Reasoning**:
The subtask requires unzipping the downloaded file. The `!unzip` command is appropriate for this.



In [ ]:
import zipfile
import os

zip_file_path = 'madis-metar-visibility-2002.zip'
extraction_path = '/content/' # Extract directly to /content/

# Check if the zip file exists
if os.path.exists(zip_file_path):
    try:
        with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
            zip_ref.extractall(extraction_path)
        print(f"Successfully unzipped {zip_file_path} to {extraction_path}")
        # List contents of the extraction path to confirm
        print("\nContents of /content/ after unzipping:")
        !ls -F /content/
    except Exception as e:
        print(f"Error unzipping file: {e}")
else:
    print(f"Zip file not found: {zip_file_path}")
    print("Please ensure the dataset is downloaded to the correct location.")

Successfully unzipped madis-metar-visibility-2002.zip to /content/

Contents of /content/ after unzipping:
01/  03/  05/  07/  09/  11/  madis-metar-visibility-2002.zip
02/  04/  06/  08/  10/  12/  sample_data/


In [ ]:
import pandas as pd
import os

# Create an empty list to store individual DataFrames
dfs = []

# Iterate through the directories from '01' to '12' (representing months)
for month in range(1, 13):
    month_str = str(month).zfill(2) # Format month as '01', '02', etc.
    month_path = os.path.join('/content', month_str)

    if os.path.exists(month_path):
        # Iterate through the files (days) inside the month's directory
        for day_file in sorted(os.listdir(month_path)):
            if day_file.endswith('.parquet'):
                file_path = os.path.join(month_path, day_file)
                try:
                    # Read each .parquet file into a pandas DataFrame
                    daily_df = pd.read_parquet(file_path)
                    # Append each daily DataFrame to the list
                    dfs.append(daily_df)
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")

# Concatenate all DataFrames in the list into a single DataFrame
if dfs:
    df = pd.concat(dfs, ignore_index=True)
    print("DataFrame loaded successfully.")

    # Display the first few rows of the DataFrame
    print("\nFirst 5 rows of the DataFrame:")
    print(df.head())

    # Print a concise summary of the DataFrame
    print("\nDataFrame Info:")
    df.info()

    # Display descriptive statistics of the DataFrame's numerical columns
    print("\nDescriptive Statistics:")
    print(df.describe())
else:
    print("No parquet files were found or loaded.")

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import os

# Create an empty list to store individual DataFrames
dfs = []

# Define the base directory where the unzipped month folders are located
base_content_path = '/content'

# Walk through the directory structure to find all .parquet files
for root, _, files in os.walk(base_content_path):
    for file in files:
        if file.endswith('.parquet'):
            file_path = os.path.join(root, file)
            try:
                # Read each .parquet file into a pandas DataFrame
                daily_df = pd.read_parquet(file_path)
                # Append each daily DataFrame to the list
                dfs.append(daily_df)
            except Exception as e:
                print(f"Error reading {file_path}: {e}")

# Concatenate all DataFrames in the list into a single DataFrame
if dfs:
    df = pd.concat(dfs, ignore_index=True)
    print("DataFrame loaded successfully.")

    # Display the first few rows of the DataFrame
    print("\nFirst 5 rows of the DataFrame (initial load):")
    print(df.head())

    # Print a concise summary of the DataFrame
    print("\nDataFrame Info (initial load):")
    df.info()

    # Display descriptive statistics of the DataFrame's numerical columns
    print("\nDescriptive Statistics (initial load):")
    print(df.describe())
else:
    print("No parquet files were found or loaded. Cannot proceed with processing.")
    # Create an empty DataFrame to avoid NameError if no files were loaded
    df = pd.DataFrame()

# --- Start of original code from cell 588c8d00 (Preprocess Data for GRU) ---

# Proceed only if the DataFrame 'df' is not empty after loading
if not df.empty:
    # Convert 'timeObs' column to datetime and rename it to 'date_time'
    df['date_time'] = pd.to_datetime(df['timeObs'])
    df = df.drop(columns=['timeObs'])

    # Create a unique identifier for each station based on latitude/longitude
    df['stationName'] = df['latitude'].astype(str) + '_' + df['longitude'].astype(str)

    # Sort by date_time for proper time-series operations before creating station_id
    # We need to ensure date_time is the index for resampling, but station_id needs to be a column.
    # So, we'll set date_time as index, and station_id remains a column to group by.
    df = df.sort_values(by=['date_time', 'stationName']).reset_index(drop=True)
    df = df.set_index('date_time')

    print("\nDataFrame after converting 'timeObs' to datetime and creating 'stationName':")
    print(df.head())
    print("\nDataFrame Info after datetime conversion:")
    df.info()
    print(f"Columns in df after stationName creation and set_index: {df.columns.tolist()}")

    original_rows = len(df)

    # Create an aggregation dictionary for numerical columns
    numeric_cols_for_agg = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    # Initialize aggregation_funcs with 'mean' for numeric columns
    aggregation_funcs = {col: 'mean' for col in numeric_cols_for_agg}

    # Group by 'stationName' and resample to hourly frequency, then aggregate using the defined functions.
    # Ensure 'stationName' is included in the groupby, and 'date_time' is the index.
    df_hourly_resampled = df.groupby('stationName').resample('H').agg(aggregation_funcs)

    print(f"\nColumns of df_hourly_resampled before reset_index: {df_hourly_resampled.columns.tolist()}")
    print(f"Index names of df_hourly_resampled before reset_index: {df_hourly_resampled.index.names}")

    # Reset index to bring 'station_id' and 'date_time' back as columns from the MultiIndex
    df_hourly = df_hourly_resampled.reset_index()

    print(f"\nColumns of df_hourly after reset_index: {df_hourly.columns.tolist()}")

    # Re-sort by station_id and date_time to ensure correct temporal order within each station for filling NaNs
    df_hourly = df_hourly.sort_values(by=['station_id', 'date_time'])

    # Handle missing values introduced by resampling or original data gaps.
    # Use forward fill (ffill) then backward fill (bfill) within each station group.
    df_hourly_filled = df_hourly.groupby('station_id').ffill().bfill().reset_index() # Added .reset_index() here

    print(f"\nColumns of df_hourly_filled after ffill/bfill: {df_hourly_filled.columns.tolist()}")

    # Ensure 'index' column is dropped if it exists (it can be created by previous reset_index if not careful)
    if 'index' in df_hourly_filled.columns:
        df_hourly_filled = df_hourly_filled.drop(columns=['index'])

    # Drop any rows that still contain NaNs after filling. These are typically rows (or entire stations)
    # that have persistent missing data for certain features even after imputation.
    df_processed = df_hourly_filled.dropna().copy() # .copy() to prevent SettingWithCopyWarning

    print("\nDataFrame after ensuring hourly frequency, handling missing values, and dropping remaining NaNs:")
    print(df_processed.head())
    print("\nDataFrame Info after processing:")
    df_processed.info()
    print(f"Columns of df_processed: {df_processed.columns.tolist()}")

    print(f"\nOriginal DataFrame rows: {original_rows}")
    print(f"Processed DataFrame rows: {len(df_processed)}")
    if original_rows != len(df_processed):
        print("Data was aggregated/resampled and missing values were handled. Some rows might have been dropped due to persistent NaNs.")
    else:
        print("Data already appears to be hourly with no significant missing values or duplicates within the same hour per station.")

    # Final check of unique stations
    print(f"\nNumber of unique stations in processed data: {df_processed['station_id'].nunique()}")
else:
    print("DataFrame 'df' is empty after loading, so no further processing was performed.")
    # Ensure df_processed is defined as an empty DataFrame if no data was loaded
    df_processed = pd.DataFrame()

DataFrame loaded successfully.

First 5 rows of the DataFrame (initial load):
   recNum  maxSkyCover stationName   latitude  longitude  elevation  \
0       0            0     b'KCJR'  38.529999 -77.870003       95.0   
1    1298            3     b'MMCM'  18.500000 -88.300003       12.0   
2    1298            4     b'MMCM'  18.500000 -88.300003       12.0   
3    1298            5     b'MMCM'  18.500000 -88.300003       12.0   
4    1299            0     b'MMCP'  19.850000 -90.550003        5.0   

              timeObs    visibility  temperature    dewpoint  windDir  \
0 2002-04-24 23:45:00  16093.500000   290.149994  276.149994      0.0   
1 2002-04-24 23:45:00  11265.450195   301.149994  299.149994    120.0   
2 2002-04-24 23:45:00  11265.450195   301.149994  299.149994    120.0   
3 2002-04-24 23:45:00  11265.450195   301.149994  299.149994    120.0   
4 2002-04-24 23:45:00  11265.450195   301.149994  296.149994    330.0   

   windSpeed      windGust      altimeter  seaLevelPress

/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1025: RuntimeWarning: overflow encountered in cast
  result = result.astype(dtype, copy=False)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1025: RuntimeWarning: overflow encountered in cast
  result = result.astype(dtype, copy=False)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce
  return umr_sum(a, axis, dtype, out, keepdims, initial, where)
/usr/local/lib/python3.12/dist-packages/pandas/core/nanops.py:1025: RuntimeWarning: overflow encountered in cast
  result = result.astype(dtype, copy

             recNum   maxSkyCover      latitude     longitude     elevation  \
count  1.414494e+08  1.414494e+08  1.414494e+08  1.414494e+08  1.414494e+08   
mean   1.367141e+03  2.500000e+00           inf           inf           inf   
min    0.000000e+00  0.000000e+00  0.000000e+00 -1.766500e+02 -3.600000e+01   
25%    6.800000e+02  1.000000e+00  3.592000e+01 -1.051200e+02  7.500000e+01   
50%    1.361000e+03  2.500000e+00  4.125000e+01 -9.302000e+01  2.340000e+02   
75%    2.042000e+03  4.000000e+00  4.530000e+01 -8.313000e+01  3.980000e+02   
max    5.082000e+03  5.000000e+00  3.402823e+38  3.402823e+38  3.402823e+38   
std    7.969656e+02  1.707825e+00           inf           inf           inf   

                             timeObs    visibility   temperature  \
count                      141448308  1.414494e+08  1.414494e+08   
mean   2002-06-30 17:15:33.907596032           inf           inf   
min              1970-01-01 00:00:00  0.000000e+00 -2.988500e+02   
25%             

KeyboardInterrupt: 

In [ ]:
print(df.columns)

Index(['recNum', 'maxSkyCover', 'stationName', 'latitude', 'longitude',
       'elevation', 'visibility', 'temperature', 'dewpoint', 'windDir',
       'windSpeed', 'windGust', 'altimeter', 'seaLevelPress', 'precip1Hour',
       'skyCover', 'skyLayerBase', 'snowCover', 'date_time'],
      dtype='object')


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout

# Ensure the data is in the correct format for TensorFlow
# Convert numpy arrays to TensorFlow tensors if they are not already
# This step might be redundant if create_sequences already outputs numpy arrays compatible with TF
# X_train_seq = tf.convert_to_tensor(X_train_seq, dtype=tf.float32)
# y_train_seq = tf.convert_to_tensor(y_train_seq, dtype=tf.float32)
# X_test_seq = tf.convert_to_tensor(X_test_seq, dtype=tf.float32)
# y_test_seq = tf.convert_to_tensor(y_test_seq, dtype=tf.float32)

# Define the GRU model
model = Sequential([
    GRU(units=50, return_sequences=True, input_shape=(look_back, X_train_seq.shape[2])),
    Dropout(0.2),
    GRU(units=50, return_sequences=False),
    Dropout(0.2),
    Dense(units=1)
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=[tf.keras.metrics.RootMeanSquaredError()])

print("GRU model defined and compiled successfully.")
model.summary()

## Load and Inspect Data

### Subtask:
Load the relevant CSV file(s) from the unzipped data into a pandas DataFrame and perform initial inspection (e.g., check data types, missing values, and columns).


**Reasoning**:
The subtask requires loading multiple parquet files, concatenating them into a single DataFrame, and then performing initial data inspection. This code block will achieve all these steps by importing necessary libraries, iterating through the directory structure, reading each parquet file, concatenating them, and finally displaying head, info, and descriptive statistics of the resulting DataFrame.



## Prepare Data for GRU

### Subtask:
Preprocess the data for GRU training, including handling missing values, converting time columns to datetime objects, setting up hourly frequency, creating sequences for the GRU model, and scaling the features. Split the data into training and testing sets.


In [ ]:
from sklearn.model_selection import train_test_split

# Sort the DataFrame by date_time and station_id to ensure chronological order for splitting
# Note: df_processed was already sorted, and df_scaled maintains that order implicitly
# We need to re-add station_id and date_time to df_scaled if we want to sort by them.
# However, for a simple chronological split of the entire dataset, we can just sort by original date_time index if it exists.
# As df_scaled only contains features and target, we will assume it's already ordered temporally correctly from previous steps.

# Define features (X) and target (y)
X = df_scaled[features]
y = df_scaled[target_variable]

# Determine the split point (e.g., 80% for training, 20% for testing)
split_ratio = 0.8
split_index = int(len(df_scaled) * split_ratio)

# Perform the chronological split
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("\nFirst 5 rows of X_train:")
print(X_train.head())
print("\nFirst 5 rows of X_test:")
print(X_test.head())

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Combine features and target for scaling
# Ensure 'date_time' is not included in the numerical features for scaling, but keep 'station_id' as a grouping key

# Drop 'date_time' and 'station_id' for scaling as they are not numerical features
# We'll re-add them or use them for splitting/grouping later if needed.
features_and_target_cols = features + [target_variable]
# Ensure that 'station_id' is also available in df_processed for later use if needed, but not included in features for scaling.
# The current features list already excludes 'station_id'.

# Filter df_processed to only include the selected features and target variable
df_to_scale = df_processed[features_and_target_cols]

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Create a copy of the dataframe for scaling to avoid SettingWithCopyWarning
df_scaled = df_to_scale.copy()

# Apply scaling to all selected numerical columns (features and target)
# For simplicity, we are applying the scaler to the entire dataset here for initial understanding,
# but for actual training, scaling should be done after train-test split, fitting on training data only.
# We will do the train-test split and proper scaling in the next step when creating sequences.

# For now, let's just show the scaling on the combined data for visualization
df_scaled[features_and_target_cols] = scaler.fit_transform(df_scaled[features_and_target_cols])

print("\nScaled DataFrame (first 5 rows):")
print(df_scaled.head())
print("\nDescriptive statistics of scaled data:")
print(df_scaled.describe())

# Store the scaler and features for later use
# In a real scenario, you'd want to save the scaler object for inverse transformation of predictions.

# Let's verify the columns after scaling
print("\nColumns in df_scaled:")
print(df_scaled.columns)

In [ ]:
print("Columns in df_processed:")
print(df_processed.columns)

# Define the target variable
target_variable = 'visibility' # Corrected from 'vsby' to 'visibility' based on df_processed columns

# Define the features to be used for the GRU model.
# Exclude 'station_id' and 'date_time' as they are not direct numerical features for the GRU input sequence.
# 'latitude' and 'longitude' are also part of 'station_id', so they will be removed as separate features.

# Identify all numerical columns
numerical_cols = df_processed.select_dtypes(include=['number']).columns.tolist()

# Columns to explicitly exclude from features
# 'recNum', 'maxSkyCover', 'windGust', 'seaLevelPress', 'precip1Hour', 'skyLayerBase'
# these columns contain placeholder values (like 3.402823e+38) or are metadata, making them unsuitable for direct use as features.
exclude_cols = ['recNum', 'maxSkyCover', 'windGust', 'seaLevelPress', 'precip1Hour', 'skyLayerBase']

features = [col for col in numerical_cols if col not in exclude_cols]

print(f"\nTarget variable: {target_variable}")
print(f"Selected features: {features}")

# Display the shape of the DataFrame after processing to verify
print(f"\nShape of df_processed: {df_processed.shape}")

### Handle Missing Values and Ensure Hourly Frequency

**Instructions:** Convert the 'date_time' column to datetime objects. If there are multiple entries for the same station and hour, aggregate to ensure hourly frequency per station (e.g., using the mean for numerical features). If the data is already hourly, proceed. If not, resample the data to an hourly frequency, grouping by `station_id`.

First, I will inspect the `date_time` column and convert it to datetime objects. Then, I will check for duplicate `station_id` and hourly `date_time` combinations. If duplicates exist, I will aggregate them, otherwise, I will proceed to the next step.

**Reasoning**:
The previous markdown block outlined the first data preprocessing step. Now, I will generate the code to implement this step, which involves converting the 'valid' column to datetime, creating a 'station_id' from 'lat' and 'lon', ensuring hourly frequency per station by resampling and aggregating, and handling missing values using forward and backward fill within each station group.



In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Combine features and target for scaling
# Ensure 'date_time' is not included in the numerical features for scaling, but keep 'station_id' as a grouping key

# Drop 'date_time' and 'station_id' for scaling as they are not numerical features
# We'll re-add them or use them for splitting/grouping later if needed.
features_and_target_cols = features + [target_variable]
# Ensure that 'station_id' is also available in df_processed for later use if needed, but not included in features for scaling.
# The current features list already excludes 'station_id'.

# Filter df_processed to only include the selected features and target variable
df_to_scale = df_processed[features_and_target_cols]

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Create a copy of the dataframe for scaling to avoid SettingWithCopyWarning
df_scaled = df_to_scale.copy()

# Apply scaling to all selected numerical columns (features and target)
# For simplicity, we are applying the scaler to the entire dataset here for initial understanding,
# but for actual training, scaling should be done after train-test split, fitting on training data only.
# We will do the train-test split and proper scaling in the next step when creating sequences.

# For now, let's just show the scaling on the combined data for visualization
df_scaled[features_and_target_cols] = scaler.fit_transform(df_scaled[features_and_target_cols])

print("\nScaled DataFrame (first 5 rows):")
print(df_scaled.head())
print("\nDescriptive statistics of scaled data:")
print(df_scaled.describe())

# Store the scaler and features for later use
# In a real scenario, you'd want to save the scaler object for inverse transformation of predictions.

# Let's verify the columns after scaling
print("\nColumns in df_scaled:")
print(df_scaled.columns)

### Create Sequences for GRU Model

**Instructions:** Create sequences from the `X_train`, `y_train`, `X_test`, and `y_test` datasets using the defined `look_back` period. Each sequence should consist of `look_back` time steps as input features (X) and the corresponding target variable (y) for the next time step. This involves iterating through the data, extracting overlapping windows, and reshaping them into a 3D format suitable for GRU input.

Since we have multiple stations, we must ensure that sequences are created independently for each `station_id` to prevent data from one station from being mixed with another, which would lead to incorrect sequence generation.

In [ ]:
import numpy as np

def create_sequences(X, y, look_back, station_ids=None):
    Xs, ys = [], []
    if station_ids is not None:
        # Group by station_id and create sequences for each station
        for station_id in station_ids.unique():
            station_indices = station_ids[station_ids == station_id].index
            station_X = X.loc[station_indices]
            station_y = y.loc[station_indices]

            if len(station_X) < look_back + 1: # Need at least look_back + 1 points for one sequence (look_back inputs + 1 target)
                continue # Skip stations with insufficient data

            for i in range(len(station_X) - look_back):
                Xs.append(station_X.iloc[i:(i + look_back)].values)
                ys.append(station_y.iloc[i + look_back])
    else:
        # Create sequences for a single continuous series (if station_ids is None)
        for i in range(len(X) - look_back):
            Xs.append(X.iloc[i:(i + look_back)].values)
            ys.append(y.iloc[i + look_back])

    return np.array(Xs), np.array(ys)

# To correctly create sequences per station, we need the 'station_id' and 'date_time' columns.
# These were dropped from df_scaled. We need to reconstruct the original df_processed to get them.

# First, re-create the df_processed with 'station_id' and 'date_time' included
# This means we need the original df_processed, not just df_scaled features.
# Let's assume df_processed (from cell 588c8d00) is available in its full form with 'station_id' and 'date_time'.

# We need the station_ids corresponding to the X_train and X_test splits.
# This means we should have carried over the 'station_id' from df_processed through the scaling process.
# To ensure this, let's re-run the scaling with 'station_id' carried along, but not scaled.

# --- Re-doing the scaling step to retain station_id for sequence creation ---

# Combine features and target for scaling
features_and_target_cols_with_station = features + [target_variable]

# Filter df_processed to only include the selected features, target variable, and station_id
# We temporarily add 'date_time' and 'station_id' to df_processed for splitting, then drop for scaling
df_temp = df_processed.copy()
df_temp = df_temp.set_index('date_time') # Ensure index is date_time for splitting

# Split df_temp chronologically into train/test, including station_id
split_index = int(len(df_temp) * split_ratio)
df_train_full = df_temp.iloc[:split_index]
df_test_full = df_temp.iloc[split_index:]

# Separate features, target, and station_id for train and test sets
X_train_unscaled = df_train_full[features]
y_train_unscaled = df_train_full[target_variable]
station_ids_train = df_train_full['stationName']

X_test_unscaled = df_test_full[features]
y_test_unscaled = df_test_full[target_variable]
station_ids_test = df_test_full['stationName']

# Initialize and fit scaler on training features and target
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_train_scaled = pd.DataFrame(scaler_X.fit_transform(X_train_unscaled), columns=features, index=X_train_unscaled.index)
y_train_scaled = pd.Series(scaler_y.fit_transform(y_train_unscaled.values.reshape(-1, 1)).flatten(), name=target_variable, index=y_train_unscaled.index)

# Transform test features and target using the *fitted* scalers
X_test_scaled = pd.DataFrame(scaler_X.transform(X_test_unscaled), columns=features, index=X_test_unscaled.index)
y_test_scaled = pd.Series(scaler_y.transform(y_test_unscaled.values.reshape(-1, 1)).flatten(), name=target_variable, index=y_test_unscaled.index)

print("Data rescaled and split for sequence creation.")

# Now, create sequences using the scaled data and station IDs

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, look_back)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, look_back)

print(f"\nShape of X_train_seq: {X_train_seq.shape}")
print(f"Shape of y_train_seq: {y_train_seq.shape}")
print(f"Shape of X_test_seq: {X_test_seq.shape}")
print(f"Shape of y_test_seq: {y_test_seq.shape}")

print("\nFirst sample of X_train_seq:")
print(X_train_seq[0])
print("\nFirst sample of y_train_seq:")
print(y_train_seq[0])

### Define Look-back Period

The look-back period, also known as the window size or sequence length, determines how many previous time steps (e.g., hours) the GRU model will observe to make a prediction for the next time step. Choosing an appropriate look-back period is crucial and often depends on the underlying patterns in the time series data.

For this task, we will start with a look-back period of 24 hours, meaning the model will consider the previous 24 hours of data to predict visibility for the next hour. This can be adjusted based on model performance and domain knowledge.

In [ ]:
# Define the look-back period (e.g., 24 hours for daily patterns)
look_back = 24

print(f"Look-back period defined as: {look_back} hours")

### Split Data into Training and Testing Sets

**Instructions:** Split the `df_scaled` DataFrame into training and testing sets. For time series data, it is critical to maintain the temporal order, so a chronological split will be performed. The `station_id` column should be treated carefully; for now, we will split the entire dataset chronologically and rely on subsequent sequence creation to handle per-station sequences.

In [ ]:
from sklearn.model_selection import train_test_split

# Sort the DataFrame by date_time and station_id to ensure chronological order for splitting
# Note: df_processed was already sorted, and df_scaled maintains that order implicitly
# We need to re-add station_id and date_time to df_scaled if we want to sort by them.
# However, for a simple chronological split of the entire dataset, we can just sort by original date_time index if it exists.
# As df_scaled only contains features and target, we will assume it's already ordered temporally correctly from previous steps.

# Define features (X) and target (y)
X = df_scaled[features]
y = df_scaled[target_variable]

# Determine the split point (e.g., 80% for training, 20% for testing)
split_ratio = 0.8
split_index = int(len(df_scaled) * split_ratio)

# Perform the chronological split
X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]
y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("\nFirst 5 rows of X_train:")
print(X_train.head())
print("\nFirst 5 rows of X_test:")
print(X_test.head())

### Select and Scale Features

**Instructions:** Select the features from the `df_processed` DataFrame that are most relevant for predicting visibility with the GRU model. Consider numerical columns like `visibility`, `temp`, `dwpt`, `rh`, `wspd`, `pres`, `p01i`, `snow` and `ice`. After selecting, scale these features using a `MinMaxScaler` or similar scaling method. Remember to fit the scaler only on the training data later, but for now, we will identify the features.

I will first identify potential feature columns by inspecting the `df_processed` DataFrame's columns and then define the target variable.

**Reasoning**:
Following the previous markdown block that explained the selection and scaling of features, I will now implement the code to identify the feature columns and the target variable ('visibility') from the `df_processed` DataFrame. This sets the stage for the subsequent scaling and sequence creation steps.



In [ ]:
print("Columns in df_processed:")
print(df_processed.columns)

# Define the target variable
target_variable = 'visibility' # Corrected from 'vsby' to 'visibility' based on df_processed columns

# Define the features to be used for the GRU model.
# Exclude 'station_id' and 'date_time' as they are not direct numerical features for the GRU input sequence.
# 'latitude' and 'longitude' are also part of 'station_id', so they will be removed as separate features.

# Identify all numerical columns
numerical_cols = df_processed.select_dtypes(include=['number']).columns.tolist()

# Columns to explicitly exclude from features
# 'recNum', 'maxSkyCover', 'windGust', 'seaLevelPress', 'precip1Hour', 'skyLayerBase'
# these columns contain placeholder values (like 3.402823e+38) or are metadata, making them unsuitable for direct use as features.
exclude_cols = ['stationName', 'date_time', 'latitude', 'longitude', target_variable, 'recNum', 'maxSkyCover', 'windGust', 'seaLevelPress', 'precip1Hour', 'skyLayerBase']

features = [col for col in numerical_cols if col not in exclude_cols]

print(f"\nTarget variable: {target_variable}")
print(f"Selected features: {features}")

# Display the shape of the DataFrame after processing to verify
print(f"\nShape of df_processed: {df_processed.shape}")

Columns in df_processed:
Index(['date_time', 'recNum', 'maxSkyCover', 'latitude', 'longitude',
       'elevation', 'visibility', 'temperature', 'dewpoint', 'windDir',
       'windSpeed', 'windGust', 'altimeter', 'seaLevelPress', 'precip1Hour',
       'skyLayerBase', 'snowCover'],
      dtype='object')

Target variable: visibility
Selected features: ['elevation', 'temperature', 'dewpoint', 'windDir', 'windSpeed', 'altimeter', 'snowCover']

Shape of df_processed: (17061955, 17)


# Task
**User Task:** Execute data preprocessing to create `df_processed` and then select features for GRU model training, ensuring `df_processed` is properly defined and features are selected.

## Execute Data Processing Cell

### Subtask:
Execute cell `588c8d00` to properly define `df_processed` and perform the necessary data preprocessing steps as outlined in the previous subtask. This will resolve the `NameError`.
